# Pan-African Soybean G×E Challenge — Submission Notebook

**Competition:** DataTour 2026 · dataafriquehub.org  
**Task:** predict soybean grain yield (kg/ha) in environments never seen in training  
**Metric:** RMSE

---

## The central lesson from this competition

Cross-validation on this dataset **does not rank models the way the leaderboard does.**
Every submission is recorded below:

| submission | model | CV (OOF) | leaderboard |
|---|---|---|---|
| original | LightGBM | 978 | **1018.89** |
| 111 | LightGBM + XGBoost | 987 | 1121.92 |
| 112 | CatBoost ensemble | 972 | 1080.25 |
| 113 | CatBoost, α=1.15 | 955 | 1069.83 |

CatBoost wins **every** cross-validation scheme tried — grouped by environment,
grouped by location, and restricted to unseen-location rows — yet it loses on the
real test by 50+ points. Its ordered target statistics over 365 `variety_id`
levels overfit in a way this data's CV structure cannot expose, because training
and validation folds always share trial series. LightGBM splits on the category
directly and does not have that failure mode.

So this notebook is built on **LightGBM**, with CatBoost held to a 20% minority
weight: enough to collect its genuine CV advantage, too little to repeat the
1069 outcome.

## Why the scores sit where they do

| quantity | value |
|---|---|
| predicting the global mean for every row | RMSE 1083 |
| between-environment std | 917 |
| within-environment std | 603 |
| environment mean predicted from climate + soil | RMSE 809, r = 0.47 |

The environment mean drives **83%** of the variance in yield, and the supplied
climate and soil rasters barely predict it. Every entry in this score range is
hitting that same data ceiling, not a modelling ceiling.

## Measured and rejected

Each of these was tried and scored, not assumed:

| idea | outcome |
|---|---|
| XGBoost in the blend | 1009 OOF, dragged the ensemble down |
| log-transform of the target | skew 0.50 → −1.02, strictly worse |
| two-stage environment-mean + deviation | 1025 vs 956 |
| variety-panel fingerprint features | 960.8 vs 959.6 |
| Finlay-Wilkinson / CV stability features | no gain |
| folds grouped by `loc` | makes `loc_mean_yield` 100% missing, while 47% of test rows sit at a known location |
| α = 1.15 dispersion scaling | **was a mistake** — tuned on a biased subset; proper CV puts the optimum at α = 1.00 |
| label-encoding `variety_id` to a float | **bug** — made the model read 365 varieties as an ordered number |
| row-order / `id` leakage | none: rank-correlation ≈ 0.05 |

Fully seeded, so re-running reproduces `submission.csv` exactly.

## 1. Setup

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
warnings.filterwarnings('ignore')

N_FOLDS, K_NBR = 5, 5
LGB_SEEDS = [42, 202, 777, 1337, 2024, 31337]
CB_SEEDS  = [42, 202, 777]
W_CB      = 0.20   # CatBoost kept a minority: strong in CV, weak on the leaderboard
ALPHA     = 1.00   # no dispersion rescaling (see header table)
np.random.seed(42)

_c = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if _c:
    TRAIN_PATH  = _c[0]
    TEST_PATH   = TRAIN_PATH.replace('train.csv', 'test.csv')
    SAMPLE_PATH = TRAIN_PATH.replace('train.csv', 'sample_submission.csv')
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = 'train.csv', 'test.csv', 'sample_submission.csv'

TARGET, ID, ENV, VAR = 'yield_kg_ha', 'id', 'environment_id', 'variety_id'
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
print('train:', TRAIN_PATH)

## 2. Load data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
gmean = train[TARGET].mean()

print(f'train {train.shape} | test {test.shape}')
print(f'environments  train {train[ENV].nunique()} | test {test[ENV].nunique()} | '
      f'overlap {len(set(train[ENV]) & set(test[ENV]))}')

em = train.groupby(ENV)[TARGET].mean()
print(f'global mean {gmean:.0f} | target std {train[TARGET].std():.0f}')
print(f'between-env std {em.std():.0f} | '
      f'within-env std {(train[TARGET] - train[ENV].map(em)).std():.0f}')
print(f'test rows at a location seen in train: '
      f'{test["loc"].isin(set(train["loc"])).mean()*100:.0f}%')

## 3. Static features

`is_irrigated` keys on `"irrig"`. `RAINFED` has three values — Rainfed,
Irrigation, Supplementary — so treating it as "anything not rainfed" wrongly
folds Supplementary in with full irrigation.

In [ ]:
def add_static(df):
    df = df.copy()
    df['SOWING_dt']   = pd.to_datetime(df['SOWING'], dayfirst=True, errors='coerce')
    df['sow_month']   = df['SOWING_dt'].dt.month
    df['sow_doy']     = df['SOWING_dt'].dt.dayofyear
    df['sow_doy_sin'] = np.sin(2 * np.pi * df['sow_doy'] / 365.25)
    df['sow_doy_cos'] = np.cos(2 * np.pi * df['sow_doy'] / 365.25)
    df['hemisphere']  = np.where(df['LAT'] >= 0, 'N', 'S')
    df['abs_lat']     = df['LAT'].abs()
    df['is_irrigated'] = (df['RAINFED'].astype(str).str.lower()
                            .str.contains('irrig')).astype(int)
    df['temp_annual_mean']   = df['wc2.1_30s_bio_1']
    df['precip_annual']      = df['wc2.1_30s_bio_12']
    df['precip_seasonality'] = df['wc2.1_30s_bio_15']
    df['temp_range']         = df['wc2.1_30s_bio_5'] - df['wc2.1_30s_bio_6']
    df['aridity_proxy']      = df['precip_annual'] / (df['temp_annual_mean'] + 10)
    for var in ['clay', 'sand', 'silt', 'nitrogen', 'phh2o', 'soc']:
        cols = [c for c in df.columns if c.startswith(var + '_')]
        if cols:
            df[f'{var}_mean_depth'] = df[cols].mean(axis=1)
    return df

train, test = add_static(train), add_static(test)
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(train, groups=train[ENV]))
print('static features done')

## 4. Leak-free target encodings

Variety and location statistics come from out-of-fold data on train (a fold never
sees its own environments) and from the full training set for test, since no test
environment appears in train.

In [ ]:
VAGG = dict(variety_mean_yield='mean', variety_median_yield='median',
            variety_std_yield='std',  variety_n_trials='count')

for c in VAGG:
    train[c] = np.nan
for fit, val in FOLDS:
    st = train.iloc[fit].groupby(VAR)[TARGET].agg(**VAGG)
    for c in st.columns:
        train.loc[train.index[val], c] = train.iloc[val][VAR].map(st[c]).values

vmed = train['variety_std_yield'].median()
train['variety_mean_yield']   = train['variety_mean_yield'].fillna(gmean)
train['variety_median_yield'] = train['variety_median_yield'].fillna(gmean)
train['variety_std_yield']    = train['variety_std_yield'].fillna(vmed)
train['variety_n_trials']     = train['variety_n_trials'].fillna(1)

test = test.merge(train.groupby(VAR)[TARGET].agg(**VAGG), on=VAR, how='left')
test['variety_mean_yield']   = test['variety_mean_yield'].fillna(gmean)
test['variety_median_yield'] = test['variety_median_yield'].fillna(gmean)
test['variety_std_yield']    = test['variety_std_yield'].fillna(vmed)
test['variety_n_trials']     = test['variety_n_trials'].fillna(0)

train['loc_mean_yield'] = np.nan
train['loc_n_trials']   = np.nan
for fit, val in FOLDS:
    st = train.iloc[fit].groupby('loc')[TARGET].agg(loc_mean_yield='mean',
                                                    loc_n_trials='count')
    for c in st.columns:
        train.loc[train.index[val], c] = train.iloc[val]['loc'].map(st[c]).values
train['loc_mean_yield'] = train['loc_mean_yield'].fillna(gmean)
train['loc_n_trials']   = train['loc_n_trials'].fillna(0)

lfull = train.groupby('loc').agg(loc_mean_yield=(TARGET, 'mean'),
                                 loc_n_trials=(TARGET, 'count'),
                                 loc_lat=('LAT', 'first'), loc_lon=('LON', 'first'))
test = test.merge(lfull[['loc_mean_yield', 'loc_n_trials']], on='loc', how='left')
test['loc_mean_yield'] = test['loc_mean_yield'].fillna(gmean)
test['loc_n_trials']   = test['loc_n_trials'].fillna(0)
print('target encodings done')

## 5. Spatial k-NN and G×E interactions

In [ ]:
def haversine(a1, o1, a2, o2):
    a1, o1, a2, o2 = map(np.radians, [a1, o1, a2, o2])
    h = np.sin((a2 - a1) / 2) ** 2 + np.cos(a1) * np.cos(a2) * np.sin((o2 - o1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

known = lfull.reset_index()

def knn_yield(lat, lon, exclude=None):
    d = haversine(lat, lon, known['loc_lat'].values, known['loc_lon'].values)
    if exclude is not None:
        d = np.where(known['loc'].values == exclude, np.inf, d)
    d = np.where(d == 0, 1e-3, d)
    i = np.argsort(d)[:K_NBR]
    return np.average(known['loc_mean_yield'].values[i], weights=1 / d[i])

# a training row excludes its own site, mirroring an unseen location at test time
train['spatial_knn_yield'] = [knn_yield(r.LAT, r.LON, r.loc) for r in train.itertuples()]
test['spatial_knn_yield']  = [knn_yield(r.LAT, r.LON)        for r in test.itertuples()]

for df in (train, test):
    df['gxe_variety_temp']    = df['variety_mean_yield'] * df['temp_annual_mean']
    df['gxe_variety_precip']  = df['variety_mean_yield'] * df['precip_annual']
    df['gxe_variety_aridity'] = df['variety_mean_yield'] * df['aridity_proxy']
print('spatial + G×E features done')

## 6. Categoricals stay categorical

Label-encoding `variety_id` to a float was the costliest bug of this competition:
the model then reads 365 varieties as an *ordered number* and splits on
meaningless thresholds. Categories are shared across train and test so codes line up.

In [ ]:
CATS = [c for c in ['COUNTRY','SEASON','RAINFED','SOURCE','COMPANY','hemisphere', VAR]
        if c in train.columns]
DROP  = [ID, TARGET, ENV, 'SOWING', 'SOWING_dt', 'loc']
feats = [c for c in train.columns if c not in DROP]

for c in CATS:
    cats = pd.Index(sorted(set(train[c].astype(str)) | set(test[c].astype(str))))
    dt = pd.CategoricalDtype(categories=cats)
    train[c] = train[c].astype(str).astype(dt)
    test[c]  = test[c].astype(str).astype(dt)

X, Xt = train[feats], test[feats]
y = train[TARGET].values
print(f'{len(feats)} features | {len(CATS)} categorical -> {CATS}')

## 7. LightGBM — the primary model

Six seeds under a fixed `GroupKFold` on `environment_id`. Seed bagging is pure
variance reduction: it cannot change the model class, so it cannot reintroduce
the CatBoost generalisation gap.

In [ ]:
LGB_P = dict(objective='regression', metric='rmse', learning_rate=0.02,
             num_leaves=15, max_depth=-1, min_data_in_leaf=50,
             feature_fraction=0.6, bagging_fraction=0.6, bagging_freq=1,
             lambda_l1=1.0, lambda_l2=3.0, verbosity=-1)

oof_lgb, te_lgb = np.zeros(len(y)), np.zeros(len(Xt))
for s in LGB_SEEDS:
    o = np.zeros(len(y))
    for fit, val in FOLDS:
        p = {**LGB_P, 'seed': s, 'bagging_seed': s,
             'feature_fraction_seed': s, 'data_random_seed': s}
        m = lgb.train(
            p, lgb.Dataset(X.iloc[fit], y[fit], categorical_feature=CATS),
            num_boost_round=5000,
            valid_sets=[lgb.Dataset(X.iloc[val], y[val], categorical_feature=CATS)],
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
        o[val]  = m.predict(X.iloc[val], num_iteration=m.best_iteration)
        te_lgb += m.predict(Xt, num_iteration=m.best_iteration) / (len(FOLDS) * len(LGB_SEEDS))
    oof_lgb += o / len(LGB_SEEDS)

print(f'LightGBM bagged OOF: {rmse(y, oof_lgb):.2f}')

## 8. CatBoost — minority hedge only

In [ ]:
Xc, Xtc = X.copy(), Xt.copy()
for c in CATS:
    Xc[c], Xtc[c] = Xc[c].astype(str), Xtc[c].astype(str)
cat_idx = [feats.index(c) for c in CATS]

oof_cb, te_cb = np.zeros(len(y)), np.zeros(len(Xt))
for s in CB_SEEDS:
    o = np.zeros(len(y))
    for fit, val in FOLDS:
        m = CatBoostRegressor(
            iterations=4000, learning_rate=0.02, depth=8, l2_leaf_reg=3,
            loss_function='RMSE', eval_metric='RMSE', random_seed=s,
            verbose=0, early_stopping_rounds=200,
            cat_features=cat_idx, one_hot_max_size=8)
        m.fit(Pool(Xc.iloc[fit], y[fit], cat_features=cat_idx),
              eval_set=Pool(Xc.iloc[val], y[val], cat_features=cat_idx),
              use_best_model=True)
        o[val] = m.predict(Xc.iloc[val])
        te_cb += m.predict(Xtc) / (len(FOLDS) * len(CB_SEEDS))
    oof_cb += o / len(CB_SEEDS)

print(f'CatBoost bagged OOF: {rmse(y, oof_cb):.2f}')

## 9. Blend, and read the honest subset

Rows whose location is absent from their own training fold are the closest
available proxy for the test set. Note that CatBoost looks better on *both*
columns here and still lost by 50 points on the real leaderboard — which is
precisely why its weight stays at 20%.

In [ ]:
newloc = np.zeros(len(y), dtype=bool)
for fit, val in FOLDS:
    newloc[val] = ~train.iloc[val]['loc'].isin(set(train.iloc[fit]['loc'])).values

oof_blend = (1 - W_CB) * oof_lgb + W_CB * oof_cb
te_blend  = (1 - W_CB) * te_lgb  + W_CB * te_cb

print(f'unseen-location rows: {newloc.sum()} / {len(y)} ({100*newloc.mean():.0f}%)\n')
print(f'{"":<22}{"all rows":>10}{"unseen-loc":>13}')
for name, o in [('LightGBM alone', oof_lgb), ('CatBoost alone', oof_cb),
                (f'blend (w_cb={W_CB})', oof_blend)]:
    print(f'  {name:<20}{rmse(y, o):>10.2f}{rmse(y[newloc], o[newloc]):>13.2f}')

print('\nalpha sweep on the blend:')
for a in [0.90, 0.95, 1.00, 1.05, 1.10, 1.15]:
    sc = gmean + a * (oof_blend - gmean)
    mark = '  <- used' if abs(a - ALPHA) < 1e-9 else ''
    print(f'  alpha={a:.2f}  all {rmse(y, sc):.1f}   unseen-loc {rmse(y[newloc], sc[newloc]):.1f}{mark}')

## 10. Write the submission

In [ ]:
final = np.clip(gmean + ALPHA * (te_blend - gmean), 0, None)

sample = pd.read_csv(SAMPLE_PATH)
tcol   = [c for c in sample.columns if c != ID][0]
sub    = sample[[ID]].copy()
sub[tcol] = sub[ID].map(dict(zip(test[ID], final)))

assert sub[tcol].isna().sum() == 0, 'missing predictions'
assert len(sub) == len(sample),     'row count differs from sample_submission'
assert list(sub.columns) == list(sample.columns), 'column layout differs'

sub.to_csv('submission.csv', index=False)
print(f'submission.csv written — {len(sub)} rows')
print(f'preds  min {final.min():.0f} | mean {final.mean():.0f} | '
      f'max {final.max():.0f} | std {final.std():.0f}')
print(sub.head().to_string(index=False))

---

## Where the remaining error lives

The environment mean is 83% of the variance and the supplied climate and soil
layers explain little of it (r = 0.47). No amount of model capacity touches that;
it is a data limit. Checked and ruled out as explanations for the top of the
leaderboard: row-order leakage, `id`-hash leakage, and shared `(loc, year)` keys
between train and test — all clean.

What would actually move the score, in order of expected value:

1. **In-season weather for the actual trial year** instead of 30-year WorldClim
   normals — rainfall and heat stress inside the real growing window. Two trials
   at one site in different years currently receive *identical* climate features
   despite different yields, so this is the single largest gap.
2. **Management intensity** — fertiliser, spacing, plot size — which separates
   high- and low-yielding trials at the same location.
3. **Maturity group / variety pedigree**, so `variety_id` generalises to the
   varieties that appear in only one or two trials.
4. **Days from sowing to the local season start**, rather than raw day of year.